# SQL Worksheet — Week3

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Filtered Student Risk Profile
Join students to attendance and class data, filter to students whose names start with `A` and whose attendance is Late or Absent, then group by student and topic. Return issue count, class date range, and a null-safe topic label.

In [0]:
--Write your code here

Select ds.student_id, ds.student_name,
COALESCE(dc.topic,'Topic is Unknown') AS topic,
MIN(dc.class_date) AS min_date,
MAX(dc.class_date) AS max_date,
SUM(CASE WHEN fa.attendance_status = 'Late' OR fa.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS issue_count

FROM rivadataplatform.dataproduct.dim_student AS ds
JOIN rivadataplatform.dataproduct.fact_attendance AS fa
ON ds.student_key = fa.student_key
JOIN rivadataplatform.dataproduct.dim_class AS dc
on fa.class_key = dc.class_key

WHERE ds.student_name like 'A%' AND fa.attendance_status IN ('Late','Present')
GROUP BY ds.student_id, ds.student_name, COALESCE(dc.topic,'Topic is Unknown')

## Question 2 — Date-Range Attendance Detail
Join attendance to students, classes, batches, and `dim_date`. Return records whose class date falls between the batch start and end dates, showing student, batch, topic, calendar day, and status. Exclude records with a null status and sort chronologically.

In [0]:
--Write your code here:

SELECT ds.student_id,
ds.student_name,
COALESCE(db.batch_name,'Batch Name Unknown') AS batch_name,
COALESCE(dc.topic,'Topic not assigned') AS topic,
COALESCE(dd.day_name,dc.class_day) AS day_name,
fa.attendance_status

FROM rivadataplatform.dataproduct.fact_attendance AS fa
JOIN rivadataplatform.dataproduct.dim_student AS ds
ON fa.student_key = ds.student_key
JOIN rivadataplatform.dataproduct.dim_class AS dc
ON fa.class_key = dc.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS db
ON fa.batch_key = db.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
ON fa.date_key = dd.date_key

WHERE dc.class_date BETWEEN db.start_date AND db.end_date
AND NULLIF(fa.attendance_status,'') IS NOT NULL

ORDER BY ds.student_id, day_name

## Question 3 — Conditional Batch Scorecard
Join attendance to batch and date dimensions and return one row per batch and calendar month. Calculate Present, Late, Absent, total records, and distinct students. Use conditional aggregation and keep only months containing at least one Absent record.

In [0]:
--Write your code here

SELECT db.batch_id, db.batch_name,
COALESCE(dd.month, extract(MONTH FROM fa.joined_at)) AS calender_month,
COUNT(DISTINCT fa.student_key) AS distinct_students,
COUNT(fa.attendance_id) AS total_records,
SUM(CASE WHEN fa.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
SUM(CASE WHEN fa.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
SUM(CASE WHEN fa.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count

FROM rivadataplatform.dataproduct.fact_attendance AS fa
JOIN rivadataplatform.dataproduct.dim_batch AS db
ON fa.batch_key = db.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
ON fa.date_key = dd.date_key

GROUP BY db.batch_id,db.batch_name, calender_month
HAVING absent_count > 0

## Question 4 — Class Coverage Including Empty Classes
Use `dim_class` as the driving table and left join attendance, students, batch, and date dimensions. Group by class and return class metadata, distinct students, total records, and average `attendance_count`, showing zero/null-safe values for classes without attendance.

 showing zero/null-safe values for classes without attendance. = means values should return proper 0 if no values


In [0]:
--Write your code here

SELECT 
dc.class_id,
COALESCE(dc.class_day,dd.day_name) AS class_day,
COALESCE(dc.topic,'Topic not assigned') AS class_topic,
COUNT(DISTINCT ds.student_id) AS distinct_students,
COUNT(fa.attendance_id) AS total_records,
COALESCE(AVG(fa.attendance_count),0) AS Avg_Attendance_Count

FROM rivadataplatform.dataproduct.dim_class AS dc
LEFT JOIN  rivadataplatform.dataproduct.fact_attendance AS fa
ON dc.class_key = fa.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_student AS ds
ON fa.student_key = ds.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS db
ON dc.batch_id = db.batch_id
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
ON fa.date_key = dd.date_key

GROUP BY dc.class_id, COALESCE(dc.class_day,dd.day_name),  class_topic

## Question 5 — Average Attendance by Topic and Status
Join attendance to class and batch dimensions. Group by batch, null-safe topic, and attendance status, and calculate average `attendance_count`, total records, and distinct students. Exclude null/blank statuses and sort by average descending.

In [0]:
--Write your code here

SELECT 
db.batch_name,
COALESCE(dc.topic,'Topic not assigned') AS class_topic,
COUNT(DISTINCT fa.student_key) AS distinct_students,
COUNT(fa.attendance_id) AS total_records,
COALESCE(AVG(fa.attendance_count),0) AS Avg_Attendance_Count,
fa.attendance_status

FROM rivadataplatform.dataproduct.fact_attendance AS fa
JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON dc.class_key = fa.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_key = fa.batch_key

GROUP BY db.batch_name, COALESCE(dc.topic,'Topic not assigned'), fa.attendance_status
HAVING NULLIF(fa.attendance_status,'') IS NOT NULL
SORT BY Avg_Attendance_Count DESC

## Question 6 — High-Volume Students With Profile Gaps
Use a `LEFT JOIN` from students through attendance, classes, and batches. Group by student and batch, then return students with at least two records, including Present count, issue count, and a null-safe phone label. Order by issue count and total records.

In [0]:
--Write your code here
SELECT
    ds.student_id,
    ds.student_name,
    COALESCE(db.batch_name, 'No batch Name') AS batch_name,
    COALESCE(NULLIF(ds.phone_no,''),'Phone Missing') AS phone_no,
    COUNT(fa.attendance_id) AS total_records,
    SUM(CASE WHEN fa.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN fa.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_count

FROM rivadataplatform.dataproduct.dim_student AS ds
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.student_key = ds.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON dc.class_key = fa.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_key = fa.batch_key

GROUP BY ds.student_id, ds.student_name, COALESCE(NULLIF(ds.phone_no,''),'Phone Missing'), COALESCE(db.batch_name, 'No batch Name')
HAVING COUNT(fa.attendance_id) >=2

ORDER BY issue_count DESC, total_records DESC;